# Train Sentinel with Hugging Face AutoTrain

Runs on a free Kaggle GPU (about 15 minutes).

**Before you start**
1. Settings (right panel): **Accelerator → GPU T4 x2**, **Internet → On**.
2. Add-ons → Secrets: add `HF_TOKEN` (your Hugging Face write token).
3. Run all cells.

What it does: builds the dataset, fine-tunes ModernBERT with AutoTrain's **default settings**, exports it for the browser (ONNX), scores it on the held-out tests, and publishes it to `bmwelu123/halo-sentinel`.

In [ ]:
HF_USERNAME = "bmwelu123"
REPO = "https://github.com/bmwelu12/halo.git"
MAKE_PUBLIC = True   # the website loads the model in the visitor's browser, so it must be public

In [ ]:
!pip -q install autotrain-advanced==0.8.36 "optimum[onnxruntime]==1.24.0"
!git clone -q $REPO halo

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_USERNAME"] = HF_USERNAME
%cd halo
!python ml/data/build.py

In [ ]:
# Fine-tune with AutoTrain (defaults). AutoTrain pushes the model to a private repo first.
!envsubst < ml/autotrain/sentinel.yml > /tmp/sentinel.yml && autotrain --config /tmp/sentinel.yml

In [ ]:
# Export to ONNX so the website can run the model in the browser (transformers.js).
from huggingface_hub import snapshot_download
local = snapshot_download(f"{HF_USERNAME}/halo-sentinel", token=os.environ["HF_TOKEN"], local_dir="halo-sentinel")
!optimum-cli export onnx --model halo-sentinel --task text-classification onnx_out
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig
q = ORTQuantizer.from_pretrained("onnx_out")
q.quantize(save_dir="onnx_q", quantization_config=AutoQuantizationConfig.avx2(is_static=False))
os.makedirs("halo-sentinel/onnx", exist_ok=True)
!cp onnx_out/model.onnx halo-sentinel/onnx/model.onnx && cp onnx_q/model_quantized.onnx halo-sentinel/onnx/model_quantized.onnx

In [ ]:
# Score on the held-out tests (fine-tuned vs keyword baseline vs untrained base).
!python ml/evaluate.py --model halo-sentinel --out halo-sentinel/eval_results.json

In [ ]:
# Upload ONNX files + eval results, then make the model public.
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
api.upload_folder(folder_path="halo-sentinel", repo_id=f"{HF_USERNAME}/halo-sentinel",
                  allow_patterns=["onnx/*", "eval_results.json"])
api.upload_file(path_or_fileobj="ml/model_card.md", path_in_repo="README.md", repo_id=f"{HF_USERNAME}/halo-sentinel")
if MAKE_PUBLIC:
    api.update_repo_settings(f"{HF_USERNAME}/halo-sentinel", private=False)
print("Done:", f"https://huggingface.co/{HF_USERNAME}/halo-sentinel")
print(open("halo-sentinel/training_params.json").read())